# Modelling your data: fields, types, and rules

The hands-on half of the schema slides (**How**, **Fields**, **Constraints**). Each
section runs one concept against the `RoadSafetyRating` model in
[`my-schema`](../my-schema/src/my_schema/models.py), then leaves you a one-cell
exercise marked **Try it**.

Some cells are *meant* to fail. A failure prints a `ValidationError`; the last lines
name the field and the rule it broke.

In [ ]:
%xmode Minimal
import json

from pydantic import TypeAdapter

from my_schema.models import RoadSafetyRating, RoadType

LINE = {"type": "LineString", "coordinates": [[-122.68, 45.52], [-122.67, 45.53]]}
POINT = {"type": "Point", "coordinates": [-122.68, 45.52]}


def feature(geometry=LINE, **properties):
    """A GeoJSON feature, as a JSON string."""
    return json.dumps({"type": "Feature", "geometry": geometry, "properties": properties})


def check(model=RoadSafetyRating, geometry=LINE, **properties):
    """Validate one feature against a model."""
    return model.model_validate_json(feature(geometry, **properties))

## One model, two shapes

The same model reads a GeoJSON feature (JSON, fields under `properties`) and a flat
row (a Python dict, fields beside `geometry`). Use `model_validate_json` for the
first and `model_validate` for the second: `model_validate` on a parsed GeoJSON
dict treats it as a flat row.

In [ ]:
check(stars=4)

In [ ]:
RoadSafetyRating.model_validate({"geometry": "LINESTRING (-122.68 45.52, -122.67 45.53)", "stars": 4})

`id` and `bbox` print as `MISSING`: absent from the input, and left out of the
output rather than written as `null`.

## Pick a base class

`Feature` gives you `id`, `bbox`, and `geometry`. `OvertureFeature`, which Overture's
own themes use, adds more.

In [ ]:
from overture.schema.base import Bathymetry
from overture.schema.system.feature import Feature

list(Feature.model_fields), list(Bathymetry.model_fields)

## Required or optional?

`stars` has no default, so it is required. Everything else is `X | None = None`.

In [ ]:
check()

**Try it:** make the cell above pass.

## Field names: aliases

The data says `LSAD`; Python says `lsad`. `Place` is `@no_extra_fields`, so data
spelled `lsad` is a key it doesn't declare, and fails.

In [ ]:
from typing import Annotated

from pydantic import Field

from overture.schema.system.model_constraint import no_extra_fields


@no_extra_fields
class Place(Feature):
    lsad: Annotated[str | None, Field(alias="LSAD")] = None


check(Place, POINT, LSAD="25").lsad

In [ ]:
check(Place, POINT, lsad="25")

**Try it:** add a `class_` field whose name in the data is `class`, and read one.

## Sized numbers

`speed_limit_kph` is a `uint16`, so its size is a bound too.

In [ ]:
check(stars=3, speed_limit_kph=70000)

## Geometry

The model allows only `LineString`.

In [ ]:
check(geometry=POINT, stars=3)

## Structs, and keys nobody declared

`Survey` is marked `@no_extra_fields`, so an undeclared key inside it fails:

In [ ]:
check(stars=3, survey={"assessor": "iRAP", "rated_by": "me"})

`RoadSafetyRating` is marked `@no_extra_fields` too, so a misspelled top-level field
fails instead of disappearing:

In [ ]:
check(stars=3, spead_limit_kph=50)

**Try it:** give `check()` a `survey` with a `surveyed_on` date, and read it back
as a `date`.

## NewTypes

A named type brings its description and checks everywhere it's used. Tested on its
own, the error header names Pydantic's internal validator, not the type.

In [ ]:
from overture.schema.system.string import CountryCodeAlpha2

country = TypeAdapter(CountryCodeAlpha2)
country.validate_python("US")

In [ ]:
country.validate_python("us")

In [ ]:
country.json_schema()

**Try it:** `country.validate_python("XX")`. What does the check actually test?

## Enums with meanings

Every legal code, and what it means.

In [ ]:
for member in RoadType:
    print(f"{member.value:10} {member.__doc__}")

In [ ]:
check(stars=3, road_type="highway")

**Try it:** write a `DocumentedEnum` for a coded column in your own data.

In [ ]:
from overture.schema.system.doc import DocumentedEnum

## Pre-built constraints

On a field:

In [ ]:
from overture.schema.system.string import HexColor, PhoneNumber, WikidataId

TypeAdapter(HexColor).validate_python("#ff0000")

In [ ]:
TypeAdapter(HexColor).validate_python("red")

**Try it:** find one good and one bad value for `PhoneNumber` and `WikidataId`.

Across fields, as a class decorator:

In [ ]:
from overture.schema.system.model_constraint import require_any_of


@require_any_of("name", "ref")
class Road(Feature):
    name: str | None = None
    ref: str | None = None


check(Road, POINT)

## Rules as data, not functions

The same rule, twice. Both reject a motorway without a speed limit.

In [ ]:
from pydantic import model_validator

from overture.schema.system.json_schema import json_schema
from overture.schema.system.model_constraint import FieldEqCondition, require_if


class AsFunction(Feature):
    road_type: str | None = None
    speed_limit_kph: int | None = None

    @model_validator(mode="after")
    def motorways_need_speed_limit(self):
        if self.road_type == "motorway" and self.speed_limit_kph is None:
            raise ValueError("speed_limit_kph is required on motorways")
        return self


@require_if(["speed_limit_kph"], FieldEqCondition("road_type", "motorway"))
class AsData(Feature):
    road_type: str | None = None
    speed_limit_kph: int | None = None


check(AsData, road_type="motorway")

Only one of them reaches JSON Schema:

In [ ]:
"if" in json.dumps(json_schema(AsFunction)), "if" in json.dumps(json_schema(AsData))

**Try it:** add `@forbid_if(["speed_limit_kph"], FieldEqCondition("road_type", "local"))`
to `AsData`, and look for it in `json_schema(AsData)`.